# Reporting Views — Financial Services Analytics
Creates SQL views: executive summary, fraud hotspots, risk heat-map, segment performance.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    transaction_date_only                       AS report_date,
    SUM(transaction_count)                      AS total_transactions,
    ROUND(SUM(total_amount), 2)                 AS total_volume,
    SUM(fraud_count)                            AS total_fraud_flagged,
    ROUND(SUM(fraud_count) / SUM(transaction_count) * 100, 3) AS overall_fraud_rate,
    ROUND(AVG(avg_amount), 2)                   AS avg_transaction_value,
    SUM(high_value_count)                       AS high_value_transactions
FROM gold_daily_transaction_summary
GROUP BY transaction_date_only
ORDER BY transaction_date_only
""")
print('Created vw_executive_summary')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_fraud_hotspots AS
SELECT
    merchant_category,
    country,
    risk_band,
    total_transactions,
    fraud_count,
    fraud_rate,
    ROUND(fraud_amount, 2) AS fraud_amount
FROM gold_fraud_analysis
WHERE fraud_rate > 0
ORDER BY fraud_rate DESC
""")
print('Created vw_fraud_hotspots')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_risk_heatmap AS
SELECT
    risk_tier,
    segment,
    account_type,
    utilisation_band,
    account_count,
    ROUND(total_balance, 2)    AS total_balance,
    ROUND(avg_balance, 2)      AS avg_balance,
    ROUND(avg_utilisation, 2)  AS avg_utilisation_pct,
    active_accounts
FROM gold_credit_risk_distribution
ORDER BY
    CASE risk_tier WHEN 'Very High' THEN 1 WHEN 'High' THEN 2 WHEN 'Medium' THEN 3 ELSE 4 END
""")
print('Created vw_risk_heatmap')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_segment_performance AS
SELECT
    segment,
    region,
    risk_tier,
    unique_customers,
    total_transactions,
    ROUND(total_volume, 2)           AS total_volume,
    ROUND(avg_transaction_value, 2)  AS avg_transaction_value,
    fraud_rate
FROM gold_segment_portfolio
ORDER BY total_volume DESC
""")
print('Created vw_segment_performance')